# 08.3 - Agents - Paperclip

Wires [Paperclip's MCP server](https://paperclip.gxl.ai/mcp) into an `aimu` model client so an agent can search, grep, and extract from 150M+ scientific papers as part of a multi-step research task.

## Notebook Setup

In [ ]:
import nest_asyncio

# Required to allow nested event loops in Jupyter notebooks
nest_asyncio.apply()

import dotenv
from genscai import paths

dotenv.load_dotenv(paths.root / "../.env")

## 01. Connect to Paperclip MCP

Paperclip exposes its CLI commands (`search`, `grep`, `cat`, `map`, `sql`) as MCP tools over an HTTP endpoint. Connect to it with `MCPClient` and inspect the available tools.

In [ ]:
from aimu.tools import MCPClient

mcp_config = {
    "mcpServers": {
        "paperclip": {"url": "https://paperclip.gxl.ai/mcp"},
    }
}

mcp_client = MCPClient(mcp_config)

for tool in mcp_client.list_tools():
    print(f"Tool: {tool.name}")
    print(f"  {tool.description}")
    print()

## 02. Direct Tool Calls

Call Paperclip tools directly to verify connectivity and explore the result format before wiring them to an agent.

In [ ]:
results = mcp_client.call_tool("search", params={"query": "malaria compartmental model", "n": 5})

for result in results:
    print(result.text)

In [ ]:
results = mcp_client.call_tool("grep", params={"pattern": "R_0.*reproduction number", "n": 5})

for result in results:
    print(result.text)

## 03. Agentic Research Task

Wire Paperclip tools into an `OllamaClient` so the agent can autonomously search, refine, and synthesize answers to a research question across the literature.

In [ ]:
from aimu.models import OllamaClient, StreamPhase
from aimu.models.ollama.ollama_client import OllamaModel

SYSTEM_MESSAGE = (
    "You are a scientific literature assistant. Use the Paperclip tools to search, "
    "grep, and extract information from papers to answer questions about infectious "
    "disease modeling. Always cite the papers you use."
)

model_client = OllamaClient(OllamaModel.MAGISTRAL_SMALL_24B, system_message=SYSTEM_MESSAGE)
model_client.mcp_client = mcp_client

for chunk in model_client.chat("Briefly introduce yourself.", stream=True):
    if chunk.phase == StreamPhase.GENERATING:
        print(chunk.content, end="", flush=True)

In [ ]:
QUERY = (
    "What modeling frameworks are most commonly used to study malaria transmission dynamics? "
    "Search the literature and summarize the key approaches."
)

for chunk in model_client.chat(QUERY, stream=True):
    if chunk.phase == StreamPhase.TOOL_CALLING:
        print(f"\n[tool: {chunk.content['name']}] -> {chunk.content['response']}\n", flush=True)
    elif chunk.phase == StreamPhase.GENERATING:
        print(chunk.content, end="", flush=True)

## 04. Inspect Results

Examine the full message thread to see how the agent used Paperclip tools across the conversation.

In [ ]:
model_client.messages

In [ ]:
del model_client